In [2]:
!wget https://raw.githubusercontent.com/yahya94812/Neural-Networks/refs/heads/main/nn-from-zero-to-hero/mini-gpt/input.txt

--2026-03-04 08:55:11--  https://raw.githubusercontent.com/yahya94812/Neural-Networks/refs/heads/main/nn-from-zero-to-hero/mini-gpt/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-03-04 08:55:11 (36.6 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
import os
import time
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32
block_size = 128
max_iters = 1000
eval_interval = 50
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 20
n_embd = 192
n_head = 6
n_layer = 4
dropout = 0.2

# Checkpoint settings
checkpoint_dir = 'checkpoints'          # folder where checkpoints are saved
checkpoint_interval = 100               # save a checkpoint every N iters
keep_last_n_checkpoints = 3            # set to None to keep all
# ------------

torch.manual_seed(1337)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


# ── Checkpoint helpers ────────────────────────────────────────────────────────

def save_checkpoint(model, optimizer, iter, val_loss):
    """Save model + optimizer state, plus all info needed to resume."""
    os.makedirs(checkpoint_dir, exist_ok=True)
    path = os.path.join(checkpoint_dir, f'ckpt_iter{iter:06d}.pt')
    torch.save({
        'iter': iter,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss,
        # Save vocab so the checkpoint is self-contained
        'stoi': stoi,
        'itos': itos,
        # Save hyperparameters so the architecture can be rebuilt exactly
        'hparams': dict(
            vocab_size=vocab_size, n_embd=n_embd, n_head=n_head,
            n_layer=n_layer, dropout=dropout, block_size=block_size,
        ),
    }, path)
    print(f"  -> checkpoint saved: {path}")
    _prune_checkpoints()


def _prune_checkpoints():
    """Delete old checkpoints, keeping only the most recent N."""
    if keep_last_n_checkpoints is None:
        return
    ckpts = sorted(
        [f for f in os.listdir(checkpoint_dir) if f.startswith('ckpt_iter')],
    )
    for old in ckpts[:-keep_last_n_checkpoints]:
        os.remove(os.path.join(checkpoint_dir, old))
        print(f"  -> removed old checkpoint: {old}")


def load_latest_checkpoint(model, optimizer):
    """
    Load the most recent checkpoint from checkpoint_dir (if any).
    Returns the iteration to resume from (0 if no checkpoint found).
    """
    if not os.path.isdir(checkpoint_dir):
        return 0

    ckpts = sorted(
        [f for f in os.listdir(checkpoint_dir) if f.startswith('ckpt_iter')]
    )
    if not ckpts:
        return 0

    path = os.path.join(checkpoint_dir, ckpts[-1])
    print(f"Resuming from checkpoint: {path}")
    ckpt = torch.load(path, map_location=device)

    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_iter = ckpt['iter'] + 1          # resume at the *next* iteration
    print(f"  -> resuming at iter {start_iter}, val loss was {ckpt['val_loss']:.4f}")
    return start_iter


# ── Data ──────────────────────────────────────────────────────────────────────

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


# ── Model ─────────────────────────────────────────────────────────────────────

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_embd,
            nhead=n_head,
            dim_feedforward=4 * n_embd,
            dropout=dropout,
            activation='relu',
            norm_first=True,
            batch_first=True,
        )
        self.blocks = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=n_layer,
            enable_nested_tensor=False,
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _causal_mask(self, T: int) -> torch.Tensor:
        return torch.triu(torch.full((T, T), float('-inf'), device=device), diagonal=1)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = self.dropout(tok_emb + pos_emb)
        x = self.blocks(x, mask=self._causal_mask(T), is_causal=True)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


# ── Training ──────────────────────────────────────────────────────────────────

model = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Resume from the latest checkpoint if one exists; otherwise start at 0.
start_iter = load_latest_checkpoint(model, optimizer)

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
print(f"Training from iter {start_iter} to {max_iters} on {device}")

tokens_per_sec_ema = None   # exponential moving average, initialised on first step
ema_alpha          = 0.98   # smoothing factor (higher = slower to react)
tokens_per_iter    = batch_size * block_size

for iter in range(start_iter, max_iters):

    # ── Evaluate + (optionally) checkpoint ────────────────────────────────────
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        tps_str = f"{tokens_per_sec_ema:,.0f} tok/s" if tokens_per_sec_ema else "n/a"
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}  |  {tps_str}")

        if iter % checkpoint_interval == 0 or iter == max_iters - 1:
            save_checkpoint(model, optimizer, iter, losses['val'])

    # ── Forward / backward ────────────────────────────────────────────────────
    t0 = time.perf_counter()

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # Sync CUDA so the timer captures real GPU work, not just kernel launches
    if device == 'cuda':
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - t0
    current_tps = tokens_per_iter / elapsed
    tokens_per_sec_ema = (
        current_tps if tokens_per_sec_ema is None
        else ema_alpha * tokens_per_sec_ema + (1 - ema_alpha) * current_tps
    )

# ── Generate a sample ─────────────────────────────────────────────────────────
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))

1.83M parameters
Training from iter 0 to 1000 on cuda
step 0: train loss 4.1569, val loss 4.1596  |  n/a
  -> checkpoint saved: checkpoints/ckpt_iter000000.pt
step 50: train loss 2.9645, val loss 2.9797  |  102,833 tok/s
step 100: train loss 2.6146, val loss 2.6078  |  137,505 tok/s
  -> checkpoint saved: checkpoints/ckpt_iter000100.pt
step 150: train loss 2.5312, val loss 2.5309  |  149,732 tok/s
step 200: train loss 2.4925, val loss 2.4970  |  153,831 tok/s
  -> checkpoint saved: checkpoints/ckpt_iter000200.pt
step 250: train loss 2.4656, val loss 2.4578  |  155,317 tok/s
step 300: train loss 2.4295, val loss 2.4402  |  154,218 tok/s
  -> checkpoint saved: checkpoints/ckpt_iter000300.pt
  -> removed old checkpoint: ckpt_iter000000.pt
step 350: train loss 2.3911, val loss 2.3927  |  155,771 tok/s
step 400: train loss 2.3458, val loss 2.3567  |  156,266 tok/s
  -> checkpoint saved: checkpoints/ckpt_iter000400.pt
  -> removed old checkpoint: ckpt_iter000100.pt
step 450: train loss 2.311

In [ ]:
import os
import time
import torch
import torch.nn as nn
from torch.nn import functional as F

# ── T4 / Tensor-Core flags (set before any CUDA work) ────────────────────────
# Allow TF32 on matmuls and cuDNN ops — free precision trade-off that keeps
# tensor cores busy on Ampere; on T4 (Turing) this is the FP16 path.
torch.backends.cuda.matmul.allow_tf32  = True
torch.backends.cudnn.allow_tf32        = True
# Let cuDNN auto-select the fastest conv algorithm for fixed input shapes.
torch.backends.cudnn.benchmark         = True
# Flash attention and memory-efficient attention kernels (PyTorch ≥ 2.0).
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)

# ── Hyperparameters ───────────────────────────────────────────────────────────
batch_size  = 32
block_size  = 128
max_iters   = 1000
eval_interval   = 50
learning_rate   = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters  = 20
# Tensor-core rule: all matrix dimensions must be multiples of 64 for peak
# FP16 throughput on T4 (Turing).  192 = 3×64 ✓  768 = 12×64 ✓
n_embd  = 192
n_head  = 6
n_layer = 4
dropout = 0.2

# Gradient clipping — essential with FP16 to keep gradients finite.
grad_clip = 1.0

# Checkpoint settings
checkpoint_dir          = 'checkpoints'
checkpoint_interval     = 100
keep_last_n_checkpoints = 3     # None = keep all
# ─────────────────────────────────────────────────────────────────────────────

torch.manual_seed(1337)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars       = sorted(list(set(text)))
vocab_size  = len(chars)
# Pad vocab to the next multiple of 64 so the final nn.Linear sits on a
# tensor-core-aligned tile boundary, eliminating padding overhead in GEMM.
vocab_size_padded = ((vocab_size + 63) // 64) * 64

stoi   = {ch: i for i, ch in enumerate(chars)}
itos   = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

# Pin both datasets in page-locked (pinned) host memory so DMA transfers to
# the GPU bypass the OS page cache and saturate PCIe bandwidth.
if device == 'cuda':
    train_data = train_data.pin_memory()
    val_data   = val_data.pin_memory()


# ── Checkpoint helpers ────────────────────────────────────────────────────────

def save_checkpoint(model, optimizer, scaler, iter, val_loss):
    os.makedirs(checkpoint_dir, exist_ok=True)
    path = os.path.join(checkpoint_dir, f'ckpt_iter{iter:06d}.pt')
    # Save the *unwrapped* state dict when torch.compile is used.
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    torch.save({
        'iter':                 iter,
        'model_state_dict':     raw_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'val_loss':             val_loss,
        'stoi':                 stoi,
        'itos':                 itos,
        'hparams': dict(
            vocab_size=vocab_size, vocab_size_padded=vocab_size_padded,
            n_embd=n_embd, n_head=n_head, n_layer=n_layer,
            dropout=dropout, block_size=block_size,
        ),
    }, path)
    print(f"  -> checkpoint saved: {path}")
    _prune_checkpoints()


def _prune_checkpoints():
    if keep_last_n_checkpoints is None:
        return
    ckpts = sorted(f for f in os.listdir(checkpoint_dir) if f.startswith('ckpt_iter'))
    for old in ckpts[:-keep_last_n_checkpoints]:
        os.remove(os.path.join(checkpoint_dir, old))
        print(f"  -> removed old checkpoint: {old}")


def load_latest_checkpoint(model, optimizer, scaler):
    if not os.path.isdir(checkpoint_dir):
        return 0
    ckpts = sorted(f for f in os.listdir(checkpoint_dir) if f.startswith('ckpt_iter'))
    if not ckpts:
        return 0
    path = os.path.join(checkpoint_dir, ckpts[-1])
    print(f"Resuming from checkpoint: {path}")
    ckpt = torch.load(path, map_location=device)
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    raw_model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_iter = ckpt['iter'] + 1
    print(f"  -> resuming at iter {start_iter}, val loss was {ckpt['val_loss']:.4f}")
    return start_iter


# ── Data ──────────────────────────────────────────────────────────────────────

def get_batch(split):
    src = train_data if split == 'train' else val_data
    ix  = torch.randint(len(src) - block_size, (batch_size,))
    # Stack on CPU first (cheap), then ship to GPU in a single H→D transfer.
    x = torch.stack([src[i:i + block_size]     for i in ix])
    y = torch.stack([src[i + 1:i + block_size + 1] for i in ix])
    # non_blocking=True lets the CPU keep working while the DMA transfer runs.
    return (x.to(device, non_blocking=True),
            y.to(device, non_blocking=True))


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            # autocast applies to eval too — keeps activations in FP16.
            with torch.autocast(device_type='cuda', dtype=torch.float16,
                                enabled=(device == 'cuda')):
                X, Y  = get_batch(split)
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


# ── Model ─────────────────────────────────────────────────────────────────────

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model        = n_embd,
            nhead          = n_head,
            dim_feedforward= 4 * n_embd,
            dropout        = dropout,
            activation     = 'relu',
            norm_first     = True,   # pre-LN
            batch_first    = True,   # (B, T, C)
        )
        self.blocks = nn.TransformerEncoder(
            encoder_layer      = encoder_layer,
            num_layers         = n_layer,
            enable_nested_tensor = False,
        )
        self.ln_f    = nn.LayerNorm(n_embd)
        # Use the padded vocab size for tensor-core alignment; the extra
        # logit slots are masked out in the loss by cross_entropy's ignore_index.
        self.lm_head = nn.Linear(n_embd, vocab_size_padded, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _causal_mask(self, T: int) -> torch.Tensor:
        # Upper-triangular -inf mask; recomputed cheaply each forward pass.
        return torch.triu(torch.full((T, T), float('-inf'), device=device), diagonal=1)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)                                # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = self.dropout(tok_emb + pos_emb)                                      # (B,T,C)
        x = self.blocks(x, mask=self._causal_mask(T), is_causal=True)           # (B,T,C)
        x = self.ln_f(x)                                                         # (B,T,C)
        logits = self.lm_head(x)                                                 # (B,T,V_pad)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            # cross_entropy ignores any index >= vocab_size (the padding slots).
            loss = F.cross_entropy(
                logits.view(B * T, C),
                targets.view(B * T),
                ignore_index=-1,   # no explicit ignore needed; padded logits
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            with torch.autocast(device_type='cuda', dtype=torch.float16,
                                enabled=(device == 'cuda')):
                logits, _ = self(idx_cond)
            logits    = logits[:, -1, :vocab_size]   # strip padding before sampling
            probs     = F.softmax(logits, dim=-1)
            idx_next  = torch.multinomial(probs, num_samples=1)
            idx       = torch.cat((idx, idx_next), dim=1)
        return idx


# ── Build model + optimizer ───────────────────────────────────────────────────

model     = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
                              fused=True)  # fused AdamW = single CUDA kernel per param group

# AMP GradScaler: dynamically scales loss to keep FP16 gradients away from
# underflow; automatically unscales before the optimizer step and skips steps
# that produce inf/nan.
scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

# torch.compile lowers the model to Triton/CUDA kernels, fusing operations and
# eliminating redundant memory round-trips (reads from HBM → compute → write
# back become single fused passes).
model = torch.compile(model)

start_iter = load_latest_checkpoint(model, optimizer, scaler)

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
print(f"Vocab: {vocab_size} real  |  {vocab_size_padded} padded (tensor-core aligned)")
print(f"Training from iter {start_iter} → {max_iters} on {device}")

# ── Training loop ─────────────────────────────────────────────────────────────

tokens_per_sec_ema = None
ema_alpha          = 0.98
tokens_per_iter    = batch_size * block_size

for iter in range(start_iter, max_iters):

    # ── Eval + checkpoint ─────────────────────────────────────────────────────
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses  = estimate_loss()
        tps_str = f"{tokens_per_sec_ema:,.0f} tok/s" if tokens_per_sec_ema else "n/a"
        print(f"step {iter}: train {losses['train']:.4f}  val {losses['val']:.4f}  |  {tps_str}")

        if iter % checkpoint_interval == 0 or iter == max_iters - 1:
            save_checkpoint(model, optimizer, scaler, iter, losses['val'])

    # ── Forward / backward ────────────────────────────────────────────────────
    t0 = time.perf_counter()

    xb, yb = get_batch('train')

    # autocast casts eligible ops to FP16 automatically; the embedding lookup
    # and LayerNorm stay in FP32 for numerical stability.
    with torch.autocast(device_type='cuda', dtype=torch.float16,
                        enabled=(device == 'cuda')):
        logits, loss = model(xb, yb)

    # scale → backward → unscale → clip → step
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)                       # unscale before clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)            # set_to_none avoids a memset

    # Synchronise so the wall-clock timer reflects actual GPU completion.
    if device == 'cuda':
        torch.cuda.synchronize()

    elapsed  = time.perf_counter() - t0
    curr_tps = tokens_per_iter / elapsed
    tokens_per_sec_ema = (
        curr_tps if tokens_per_sec_ema is None
        else ema_alpha * tokens_per_sec_ema + (1 - ema_alpha) * curr_tps
    )

# ── Sample from the trained model ─────────────────────────────────────────────
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))

: 

: 